In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import joblib
import re
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import sys

from sklearn.model_selection import train_test_split
from plotly.subplots import make_subplots
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.feature_extraction.text import CountVectorizer   # Sac de mots
from sklearn.feature_extraction.text import TfidfVectorizer    # TF-IDF
from sklearn.feature_extraction.text import (
    ENGLISH_STOP_WORDS  #Stop words English
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
import pickle
import nltk
import shutil
import os
import glob
import warnings

import optuna
import joblib

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import (
    accuracy_score,  # Précision globale
    precision_score,  # Précision
    recall_score,  # Rappel
    f1_score,  # F1-Score
    confusion_matrix,  # Matrice de confusion
    roc_auc_score,  # AUC (Area Under the Curve)
    roc_curve,  # Courbe ROC
    classification_report,  # Rapport de classification
    mean_squared_error,  # Erreur quadratique moyenne (MSE)
    mean_absolute_error,  # Erreur absolue moyenne (MAE)
    r2_score,  # Coefficient de détermination R²
    auc  # Calcul de l'AUC
)
from sklearn.exceptions import ConvergenceWarning

# Librairies pour LDA
import gensim                    # Modélisation des topics et NLP
from gensim import corpora       # Création du dictionnaire de mots
from gensim.models import LdaModel  # Modèle LDA simple
from gensim.utils import simple_preprocess  # Prétraitement simple des textes
from gensim.models import CoherenceModel    # Calcul de la cohérence des topics
from gensim.models import Phrases           # Création de bigrammes/trigrammes
from gensim.models.phrases import Phraser   # Optimisation bigrammes/trigrammes
from gensim import corpora, models          # Utilisation des modèles
from gensim.models import LdaMulticore      # Modèle LDA parallèle
import multiprocessing                      # Gestion du multi-threading


nltk_data_dir = './nltk_data'


# On laisse au cas où
# # Aggressively clean corrupted NLTK data
# if os.path.exists(nltk_data_dir):
#     shutil.rmtree(nltk_data_dir)
# # Also remove any corrupted zip files from system paths
# for path in nltk.data.path:
#     if os.path.exists(path):
#         for zipfile in glob.glob(os.path.join(path, '*.zip')):
#             try:
#                 os.remove(zipfile)
#             except:
#                 pass

# Create fresh nltk_data directory
os.makedirs(nltk_data_dir, exist_ok=True)
nltk.data.path.insert(0, nltk_data_dir)

# Download resources with fresh start
print("Downloading NLTK resources...")
nltk.download('wordnet', download_dir=nltk_data_dir, quiet=True)
nltk.download('omw-1.4', download_dir=nltk_data_dir, quiet=True)
nltk.download('wordnet_ic', download_dir=nltk_data_dir, quiet=True)
nltk.download('averaged_perceptron_tagger', download_dir=nltk_data_dir, quiet=True)
print("NLTK resources downloaded successfully!")

from nltk.corpus import wordnet
from nltk.stem.snowball import SnowballStemmer

from contraction_fix import fix as expand_contractions # Contraction

In [ ]:
import spacy                                # NLP avancé
from spacy import displacy                  # Visualisation
from spacy.cli import download
download("en_core_web_sm")

In [ ]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
def stop_words(text):
    # Normalisation simple des apostrophes typographiques
    text = text.replace("’", "'")
    tokens = text.split()
    expanded = []
    for t in tokens:
        key = t.lower()
        if key not in ENGLISH_STOP_WORDS:
            expanded.append(t)
    return " ".join(expanded)

text = "I'm happy but I don't know why."
print(text)
print(stop_words(expand_contractions(text)))

In [ ]:
stemmer = nltk.stem.porter.PorterStemmer()
lemmatizer = nltk.stem.WordNetLemmatizer()

def stemmerLemma(text):
    # Normalisation simple des apostrophes typographiques
    text = text.replace("’", "'")
    expanded = []
    t2 = nlp(text)
    
    for t in t2:
        
        key = t.text.lower()
        # print(key, t.pos_, stemmer.stem(key), lemmatizer.lemmatize(key))
        if t.pos_ == "VERB":
            result = stemmer.stem(key)
        elif t.pos_ == "NOUN" or t.pos_ == "ADJ" or t.pos_ == "PROPN":
            result = lemmatizer.lemmatize(key)
        else:
            result = key
        expanded.append(result)
    return " ".join(expanded)

In [ ]:
emoticons_str = r"""
    (?:
        [:=;] # Eyes
        [oO\-]? # Nose (optional)
        [D\)\]\(\]/\\OpP] # Mouth
    )"""
html_str = r"<[^>]+>"
mentions_str = r"(?:@[\w_]+)"
hashtags_str = r"(?:\#+[\w_]+[\w\'_\-]*[\w_]+)"
url_str = r"http[s]?://(?:[a-z]|[0-9]|[$-_@.&amp;+]|[!*\(\),]|(?:%[0-9a-f][0-9a-f]))+"
number_str = r"(?:(?:\d+,?)+(?:\.?\d+)?)"
compose_str = r"(?:[a-z][a-z'\-_]+[a-z])"
mots_str = r"(?:[\w_]+)"
reste_str = r"(?:[\S]+)"

total_str = [
    emoticons_str,
    html_str,
    mentions_str,
    hashtags_str,
    url_str,
    number_str,
    compose_str,
    mots_str,
    reste_str,
]

emoticons_regex = re.compile(emoticons_str, re.VERBOSE | re.IGNORECASE)
html_regex = re.compile(html_str, re.VERBOSE | re.IGNORECASE)
mentions_regex = re.compile(mentions_str, re.VERBOSE | re.IGNORECASE)
hashtags_regex = re.compile(hashtags_str, re.VERBOSE | re.IGNORECASE)
url_regex = re.compile(url_str, re.VERBOSE | re.IGNORECASE)
number_regex = re.compile(number_str, re.VERBOSE | re.IGNORECASE)
compose_regex = re.compile(compose_str, re.VERBOSE | re.IGNORECASE)
mots_regex = re.compile(mots_str, re.VERBOSE | re.IGNORECASE)
reste_regex = re.compile(reste_str, re.VERBOSE | re.IGNORECASE)
total_regex = re.compile(r'('+'|'.join(total_str)+')', re.VERBOSE | re.IGNORECASE)

def preprocess(s, emoticons=False, html=False, mentions=False, hashtags=False, url=False, number=False, compose=True,
               stemmerlemma=True, stopwords=True):
    tokens = total_regex.findall(s)

    tokens = [tok for tok in tokens if 
        mots_regex.search(tok) and # TODO: fix
        (emoticons or not emoticons_regex.search(tok)) and
        (html or not html_regex.search(tok)) and
        (mentions or not mentions_regex.search(tok)) and
        (hashtags or not hashtags_regex.search(tok)) and
        (url or not url_regex.search(tok)) and
        (number or not number_regex.search(tok)) and
        (compose or not compose_regex.search(tok))
    ]
            
    sentence = " ".join(tokens)
    if stemmerlemma:
        sentence = stemmerLemma(sentence)
    if stopwords:
        sentence = stop_words(sentence)
        
    return sentence

preprocess("This is an example, ! of #CountVectorizer for creating a vector https://leotta.ro 런쥔을공평하게_대하세요")
# print(preprocess("This is another example of CountVectorizer"))
# print(preprocess("with or without parameters"))

# Répartition des données (upsampling / downsampling / rien)

In [ ]:
#sampling="nothing"
#sampling="up"
# sampling="down"
sampling="weighted"

column="science_related"
# column="scientific_claim"
# column="scientific_reference"
# column="scientific_context"

# ---------------------------------------------------------------
# ------------------------- PAS TOUCHER -------------------------
# ---------------------------------------------------------------

df_lue=pd.read_csv('scitweets_export.tsv', sep='\t')
print(f"df d'origine : {len(df_lue)} lignes")

# 0. ça aide
if column=="science_related":
    tache_res_display = ["pas scientifique", "scientifique"]
if column=="scientific_claim":
    tache_res_display = ["pas claim", "claim"]
if column=="scientific_reference":
    tache_res_display = ["pas référence", "référence"]
if column=="scientific_context":
    tache_res_display = ["pas context", "context"]

# 1. Si on spécifie une colonne précisée scientifique, on ne travaille que sur la df des tweets scientifiques
if column != "science_related":
    df_lue = df_lue[df_lue["science_related"] == 1].reset_index(drop=True)
    print(f"df filtrée : {len(df_lue)} lignes")
    
# 2. Séparer le jeu de donné
texte = [preprocess(t) for t in df_lue.text]
X_train_related, X_test_related, y_train_related, y_test_related = train_test_split(
    texte, df_lue[column], test_size=0.2, stratify=df_lue[column], random_state=42
)

X = X_train_related + X_test_related
y = list(y_train_related) + list(y_test_related)
print(f"Train sur {len(X_train_related)} lignes, test sur {len(X_test_related)}")

# Benchmark des models

In [ ]:
# 1. Fonction pour sample un x et un Y.
def sample_data(X, y):
    #y_series = pd.Series(y if isinstance(y, list) else y.values)
    y_series = pd.Series(y)
    X_series = pd.Series(X)

    idx_min = y_series[y_series == 0].index
    idx_max = y_series[y_series == 1].index
    if len(idx_min) > len(idx_max):
        idx_min, idx_max = idx_max, idx_min

    nb_lignes_min = len(idx_min)
    nb_lignes_max = len(idx_max)

    if sampling == "up":
        idx_min = idx_min.to_series().sample(n=nb_lignes_max, replace=True, random_state=42).index
    elif sampling == "down":
        idx_max = idx_max.to_series().sample(n=nb_lignes_min, random_state=42).index

    idx_final = pd.Series(idx_min.tolist() + idx_max.tolist()).sample(frac=1, random_state=42).values
    return X_series[idx_final].tolist(), y_series[idx_final].tolist()

# 2. Liste de tes modèles (sans GaussianNB, remplacé par MultinomialNB)
if sampling == "weighted":
    models = [
        ("KNN", "cornflowerblue", None),
        ("DecisionTree", "orange", None),        
        ("MultinomialNB (fit_prior=True)", "limegreen", MultinomialNB(fit_prior=True)),
        ("LinearSVC", "firebrick", LinearSVC(class_weight="balanced")),
        ("RandomForest", "darkorchid", RandomForestClassifier(class_weight='balanced'))
    ]
else:
    models = [
        ("KNN", "cornflowerblue", KNeighborsClassifier()),
        ("DecisionTree", "orange", DecisionTreeClassifier()),
        ("MultinomialNB (fit_prior=False)", "limegreen", MultinomialNB(fit_prior=False)),
        ("LinearSVC", "firebrick", LinearSVC()),
        ("RandomForest", "darkorchid", RandomForestClassifier())
    ]

metriques = ["accuracy", "f1_macro"]

names = []
colors = []
results = {m:[] for m in metriques}

texte = [preprocess(t) for t in df_lue.text]
X_train_related = texte
y_train_related = df_lue[column].values
print(len(X_train_related), len(y_train_related))

seed = 42
kfold = KFold(n_splits=10, random_state=seed, shuffle=True)

print("Début de l'évaluation des modèles...\n")

for name, color, model in models:
    if model is None:
        continue
            
    names.append(name)
    colors.append(color)
    print(f"{"-"*15} Modèle : {name} {"-"*15}")
    fold_accuracies = []
    fold_f1_macros = []

    # On sample les données de test et de train séparément. On sample sur les indices pour garder les données X et Y synchronisées
    for fold, (fold_train_idx, fold_val_idx) in enumerate(kfold.split(X_train_related, y_train_related)):
        X_fold_train = [X_train_related[i] for i in fold_train_idx]
        y_fold_train = y_train_related[fold_train_idx]
        
        X_fold_val = [X_train_related[i] for i in fold_val_idx]
        y_fold_val = y_train_related[fold_val_idx]

        if sampling == "up" or sampling == "down":
            X_fold_train_resampled, y_fold_train_resampled = sample_data(X_fold_train, y_fold_train)
        else:
            X_fold_train_resampled, y_fold_train_resampled = X_fold_train, y_fold_train
    
        # la pipeline utilise nos donnés samplés
        pipeline = Pipeline([
            ("vectorizer", TfidfVectorizer(lowercase=False, ngram_range=(1, 2))),
            ("model", model)
        ])
        pipeline.fit(X_fold_train_resampled, y_fold_train_resampled)
            
        # On stocke les scores du pli
        y_pred_val = pipeline.predict(X_fold_val)
        fold_accuracies.append(accuracy_score(y_fold_val, y_pred_val))
        fold_f1_macros.append(f1_score(y_fold_val, y_pred_val, average="macro"))

    print(f"Accuracy Moyenne (10-Fold) : {np.mean(fold_accuracies):.4f}")
    print(f"F1-Macro Moyen  (10-Fold) : {np.mean(fold_f1_macros):.4f}\n")
    results["accuracy"].append(np.mean(fold_accuracies))
    results["f1_macro"].append(np.mean(fold_f1_macros))

In [ ]:
# for metrique in metriques:
#     fig = plt.figure()
#     fig.suptitle("Comparaison des " + metrique)
#     ax = fig.add_subplot(111)
#     plt.boxplot(results[metrique])
#     ax.set_xticklabels(names)
#     plt.show()

x = np.arange(len(metriques))
width = 0.15
fig, ax = plt.subplots(figsize=(12, 6))
for i, name in enumerate(names):
    model_scores = [np.mean(results[m][i]) for m in metriques]
    ax.bar(x + (i - (len(names) - 1) / 2) * width, model_scores, width, label=name, color=colors[i])
ax.set_title(f"Comparaison des modèles par métrique pour {column}" + (f" ({sampling})" if sampling != "nothing" else ""))
ax.set_xlabel("Métriques")
ax.set_ylabel("Score moyen (CV)")
ax.set_xticks(x)
ax.set_xticklabels(metriques)
ax.set_ylim(0, 1)
ax.legend(title="Modèles", loc="lower center")
ax.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

filename = f"comparison_modeles_{column}_{sampling}.png" if sampling != "nothing" else f"comparison_modeles_{column}.png"
fig.savefig(filename, dpi=300, bbox_inches="tight")
print(f"Plot saved to: {filename}")

# Comparaison poussée entre weighted LinearSVC et Normal LinearSVC

In [ ]:
def plot_confusion_matrix(cm, classes, title='Matrice de confusion', cmap=plt.cm.Blues):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, xticklabels=classes, yticklabels=classes)
    plt.title(title)
    plt.ylabel('Vérité terrain')
    plt.xlabel('Prédictions')
    plt.tight_layout()
    plt.show()

sampling = "nothing" 
column = "science_related"
df_lue = pd.read_csv('scitweets_export.tsv', sep='\t')
print(f"df d'origine : {len(df_lue)} lignes")

# Configuration de l'affichage
tache_res_display = ["Classe 0 (Non-Sci)", "Classe 1 (Science)"]

# 1. Préparation du texte complet
texte = [preprocess(t) for t in df_lue.text]

# 2. Séparation stricte et stratifiée Train / Test (80% / 20%)
X_train_brut, X_test_brut, y_train, y_test = train_test_split(
    texte, df_lue[column], test_size=0.2, stratify=df_lue[column], random_state=42
)

#vectorisation (TF-IDF) sur les données brutes
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=7000, lowercase=False)
X_train_vec = vectorizer.fit_transform(X_train_brut)
X_test_vec  = vectorizer.transform(X_test_brut)

# entrainement des modèles
model_rien = LinearSVC(C=1.0, random_state=42)
model_rien.fit(X_train_vec, y_train)

model_balanced = LinearSVC(C=1.0, class_weight="balanced", random_state=42)
model_balanced.fit(X_train_vec, y_train)

# prédictions sur le test
preds_rien = model_rien.predict(X_test_vec)
preds_balanced = model_balanced.predict(X_test_vec)

def analyser_comportement(y_vrai, y_pred, nom_modele):
    cm = confusion_matrix(y_vrai, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    total_non_science = tn + fp
    total_science = fn + tp
    
    taux_detection_science = (tp / total_science) * 100
    tweets_science_rates = fn
    
    print(f" STATISTIQUES COMPORTEMENTALES : {nom_modele.upper()} ")
    plot_confusion_matrix(cm, classes=["non-scientifique", "scientifique"], title=f"Matrice de confusion - {nom_modele}")
    print(f"Sur {total_science} VRAIS tweets de Science :")
    print(f" Vrais Positifs : {tp}")
    print(f" Faux Négatifs): {tweets_science_rates}")
    print(f" Rappel sciences : {taux_detection_science:.1f}%")
    print("Détail Précision / Rappel par classe :")
    print(classification_report(y_vrai, y_pred, target_names=["Classe 0 (Non-Sci)", "Classe 1 (Science)"]))
    print("\n")

# Lancement de l'analyse
analyser_comportement(y_test, preds_rien, "Modèle Sans Rien (De Base)")
analyser_comportement(y_test, preds_balanced, "Modèle Weight Balanced")

# Recherche des meilleurs hyperParamètres LiearSVC

In [ ]:
def objective_linear(trial):
    ngram_max = trial.suggest_int("ngram_max", 2, 4)
    param_ngram_range = (1, ngram_max)
    param_max_features = trial.suggest_int("max_features", 8000, 20000, step=500)

    param_min_df = trial.suggest_int("min_df", 1, 4)

    param_max_df = trial.suggest_float("max_df", 0.70, 0.99)

    param_C = trial.suggest_float("C", 0.01, 2, log=True)

    param_max_iter = trial.suggest_int("max_iter", 5000, 20000, step=1000)

    param_penalty = trial.suggest_categorical("penalty", ["l1", "l2"])

    if param_penalty == "l1":
        param_loss = "squared_hinge"
        param_dual = False
    else:
        param_loss = trial.suggest_categorical("loss", ["hinge", "squared_hinge"])
        param_dual = True if param_loss == "hinge" else False

    vectorizer = TfidfVectorizer(
        ngram_range=param_ngram_range,
        max_features=param_max_features,
        min_df=param_min_df,
        max_df=param_max_df,
        lowercase=False,
    )

    model = LinearSVC(
        C=param_C,
        penalty=param_penalty,
        loss=param_loss,
        max_iter=param_max_iter,
        class_weight="balanced",
        dual=param_dual,
        random_state=42,
    )

    pipeline = Pipeline([("vectorizer", vectorizer), ("model", model)])

    # ---------------------------------------------------------
    # 4. ÉVALUATION (CROSS-VALIDATION)
    # ---------------------------------------------------------
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)

    scores = cross_val_score(
        pipeline, X, y, cv=kfold, scoring="f1_macro", n_jobs=-1
    )

    return scores.mean()

def saveLinearSVCModel(colonne, path):
    df = pd.read_csv('scitweets_export.tsv', sep='\t')
    df_test = df.sample(n=10, random_state=0)
    df = df.drop(df_test.index)
    
    X = [preprocess(t) for t in df["text"]]
    y = df[colonne].values
    
    # Meilleurs paramètres Optuna
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_linear, n_trials=100)
    
    pipeline_final = Pipeline([
        ("vectorizer", TfidfVectorizer(
            ngram_range=(1, study.best_params['ngram_max']),
            max_features=study.best_params['max_features'],
            min_df=study.best_params['min_df'],
            max_df=study.best_params['max_df'],
            lowercase=False
            )),
        ("model", LinearSVC(
            C=study.best_params['C'],
            penalty=study.best_params['penalty'],
            loss=study.best_params['loss'],
            max_iter=study.best_params['max_iter'],
            class_weight="balanced", # mets du poids sur les erreurs de la classe minoritaire pour qu'il cherche vraiment à les corriger
            random_state=42
        ))
    ])

    # Entraînement du pipeline sur toutes les données disponibles
    pipeline_final.fit(X, y)
    
    # Enregistrement model
    joblib.dump(pipeline_final, path)
    print(f"Modèle sauvegardé sous : {path}")

# Recherche des meilleurs hyperParamètres MultinomialMB

In [ ]:
# TODO:

# Recherche des meilleurs hyperParamètres DecisionTree

In [ ]:
# TODO:

# Création et enregistrement des models

In [ ]:
saveLinearSVCModel("science_related", "ScienceRelatedModel.joblib")


# Chargement et Predict

In [ ]:
# Rechargement du modèle
model_filename = "ScienceRelatedModel.joblib"
loaded_model = joblib.load(model_filename)

# Prédictions
loaded_model.predict(df2["text"])